# Week 2 Monday — Real-time deployment

Deploys the approved churn model (v1, `churn-xgboost-models`) as a live SageMaker endpoint.

In [ ]:
import sagemaker
print(sagemaker.__version__)  # should print 2.257.5

from sagemaker import ModelPackage

role = "arn:aws:iam::666258711441:role/service-role/AmazonSageMaker-ExecutionRole-20260722T143599"
BUCKET = "beant-mlops-portfolio-666258711441"

# Approved model version 1
model_package_arn = "arn:aws:sagemaker:ca-central-1:666258711441:model-package/churn-xgboost-models/1"

model = ModelPackage(model_package_arn=model_package_arn, role=role)

## Deploy

Note: `ml.t3.medium` is **not** a valid inference instance type (learned this the hard way on Thursday) — using `ml.t2.medium`, which matches the `inference_instances` list declared when the model was registered.

In [ ]:
model.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    endpoint_name="churn-detection-endpoint",
)

# ModelPackage.deploy() doesn't reliably return a usable Predictor (no predictor_cls set) -
# construct it explicitly from the endpoint name instead.
from sagemaker.predictor import Predictor
predictor = Predictor(endpoint_name="churn-detection-endpoint")

## Test with real held-out samples

Pulls a few rows from the processed test set (same S3 location used in training) and compares predictions to actual labels.

In [ ]:
import boto3
import pandas as pd
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=BUCKET, Key="processed/test/test.csv")
test_df = pd.read_csv(obj["Body"], header=None)

sample = test_df.sample(5, random_state=42)
y_true = sample.iloc[:, 0].values
X_sample = sample.iloc[:, 1:]

result = predictor.predict(X_sample.values)
print("Predicted probabilities:", result)
print("Actual labels:          ", y_true)

## Cost note

This live endpoint bills **continuously per hour** while it exists, regardless of request volume. Per the plan, it should be deleted at the end of the week if not actively demoing:

```python
predictor.delete_endpoint()
```

(Not run here — deletion is a later step, after batch inference (Tuesday) and monitoring setup (Wednesday) have both used this same endpoint.)